# 📦 Step 1 — Create a Huihui Qwen3.8 27B Kaggle Dataset

Optional: download the exact model and matching vision projector from [huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF](https://huggingface.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF), then create or update a **private dataset under your Kaggle account**. The server can also download these files directly without a model dataset.

| File | Size | Purpose |
|---|---|---|
| `Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf` | 16.55 GB / 15.42 GiB | Main GGUF (`UD-DW-Q4_K_M`) |
| `mmproj-model-bf16.gguf` | 0.93 GB / 0.87 GiB | Matching BF16 vision projector |

Both files use revision `c4b90228246aef7ecc6cb2be9d80575a911a0f3a`, matching `2_run_server.ipynb`. The projector is needed for image input; the server can run text-only with `ENABLE_VISION=False`.

### Prerequisites

- **Internet ON**; no GPU needed for this download notebook.
- Set your **Kaggle username** in Cell 2; GitHub and Kaggle usernames can differ.
- Authenticate the Kaggle CLI for dataset creation/versioning. Supplying a username alone does not authenticate it. Do not put API tokens in a committed notebook.
- Check free local disk space (about 17.48 GB plus overhead) and your Kaggle dataset storage quota.


## ⚙️ Cell 1 — Install dependencies

Install the official Hugging Face download library and Kaggle CLI. Current `huggingface_hub` uses its supported download backend; no legacy `hf_transfer` flag is needed. See the [official download guide](https://huggingface.co/docs/huggingface_hub/guides/download).


In [ ]:
%pip install -q -U huggingface_hub kaggle


## ⚙️ Cell 2 — Configuration

Files download to a model-specific temporary directory before dataset upload. Temporary files do not survive a fresh Kaggle session; the private dataset is the reusable copy. Set your Kaggle username here and keep the source revision in sync with the server notebook if you change it.


In [ ]:
import os
import json
from pathlib import Path

KAGGLE_USERNAME = ""  # Your Kaggle username; may differ from your GitHub username
REPO_ID = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
MODEL_REVISION = "c4b90228246aef7ecc6cb2be9d80575a911a0f3a"
MODEL_FILE = "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-model-bf16.gguf"
EXPECTED_SIZES = {MODEL_FILE: 16551316384, MMPROJ_FILE: 931145888}
DOWNLOAD_DIR = Path("/tmp/huihui-qwen38-27b-model")
DATASET_SLUG = "huihui-qwen38-27b-gguf"

KAGGLE_USERNAME = KAGGLE_USERNAME or os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Set KAGGLE_USERNAME in this configuration cell.")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source: {REPO_ID} @ {MODEL_REVISION}")
print(f"Dataset: {KAGGLE_USERNAME}/{DATASET_SLUG} (private)")
print(f"Download directory: {DOWNLOAD_DIR}")


## 📥 Cell 3 — Download the two exact files

Use `hf_hub_download` with exact filenames and a pinned revision. Wildcards could select another quantization or multiple large files. Downloads go directly from Hugging Face to Kaggle; transfer time depends on the connection.


In [ ]:
from huggingface_hub import hf_hub_download

for filename in (MODEL_FILE, MMPROJ_FILE):
    print(f"Downloading {filename}...", flush=True)
    hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        revision=MODEL_REVISION,
        local_dir=str(DOWNLOAD_DIR),
    )
print("✅ Download complete.")


## 🔍 Cell 4 — Verify downloaded files

Require both exact filenames, sizes from the pinned snapshot and GGUF file headers. Reject unexpected top-level files before uploading so a reused directory cannot silently publish unrelated weights. Download metadata lives under `.cache/` and is skipped by the upload's `--dir-mode skip` option.


In [ ]:
total_bytes = 0
for filename, expected_size in EXPECTED_SIZES.items():
    path = DOWNLOAD_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")
    size = path.stat().st_size
    if size != expected_size:
        raise ValueError(f"Unexpected size for {filename}: {size}; expected {expected_size}")
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            raise ValueError(f"Invalid GGUF header: {filename}")
    total_bytes += size
    print(f"{filename}: {size / 10**9:.2f} GB ({size / 1024**3:.2f} GiB)")

unexpected = [p.name for p in DOWNLOAD_DIR.iterdir()
              if p.is_file() and p.name not in {*EXPECTED_SIZES, "dataset-metadata.json"}]
if unexpected:
    raise RuntimeError(f"Unexpected files in download directory: {unexpected}. Use a clean directory.")
print(f"✅ Both GGUFs verified: {total_bytes / 10**9:.2f} GB total")


## 📦 Cell 5 — Create or update your private Kaggle dataset

Generate dataset metadata using the username from Cell 2. Create a private dataset on the first run, or upload a new version when it already exists. Kaggle CLI authentication and sufficient account storage are required. An upload error stops execution; the final success instructions apply only after a successful upload.


In [ ]:
import subprocess

dataset_id = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
metadata = {
    "title": "Huihui Qwen3.8 27B abliterated UD-DW-Q4_K_M GGUF",
    "id": dataset_id,
    "licenses": [{"name": "apache-2.0"}],
    "isPrivate": True,
}
(DOWNLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

check = subprocess.run(
    ["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
)
if check.returncode == 0:
    command = ["kaggle", "datasets", "version", "-p", str(DOWNLOAD_DIR),
               "-m", f"Huihui model snapshot {MODEL_REVISION}", "--dir-mode", "skip"]
else:
    # Creation is private by default; do not pass --public.
    command = ["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR),
               "--dir-mode", "skip"]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset upload failed. Check Kaggle CLI authentication, username and quota.")
print(f"✅ Dataset ready: https://www.kaggle.com/datasets/{dataset_id}")


## ✅ Done

After a successful upload, open `2_run_server.ipynb`:

1. Choose **Add Input / Add data → Your Datasets**.
2. Search `huihui-qwen38-27b-gguf` and select **your own private dataset** (or the slug you configured), using the newest version.
3. Attach your current `llama-cpp-bin` dataset as well.
4. Set `DATASET_NAME` if you customized the slug. The server finds exact files in current owner/dataset mounts or older slug-only mounts.

The dataset is `<your-kaggle-username>/huihui-qwen38-27b-gguf`; this is created under your account, not supplied by a public publisher. `KAGGLE_USERNAME` is optional in the server when inputs are unambiguous. Set `DOWNLOAD_IF_MISSING=False` there to require attached model files.
